In [ ]:
import pandas as pd


mobapp_act['calc_date'] = pd.to_datetime(
    mobapp_act['calc_date']
)

subscr_status['subscr_date_act'] = pd.to_datetime(
    subscr_status['subscr_date_act']
)


# минимальная дата просмотра оффера
first_view = (
    mobapp_act[mobapp_act['metric_id'] == 1]
    .groupby('magnit_id', as_index=False)
    .agg(
        first_view_date=('calc_date', 'min')
    )
)


# минимальная дата подписки
subscr_min = (
    subscr_status
    .groupby('magnit_id', as_index=False)
    .agg(
        first_subscr_date=('subscr_date_act', 'min')
    )
)


# сравниваем просмотр и подписку
view_subscr = (
    first_view
    .merge(
        subscr_min,
        on='magnit_id',
        how='left'
    )
)


# не подписался после просмотра
view_subscr['not_converted'] = (
    view_subscr['first_subscr_date'].isna()
    | (
        view_subscr['first_subscr_date']
        <= view_subscr['first_view_date']
    )
)


# агрегаты
total_view_clients = (
    view_subscr['magnit_id']
    .nunique()
)

not_converted_clients = (
    view_subscr[
        view_subscr['not_converted']
    ]['magnit_id']
    .nunique()
)

converted_clients = (
    total_view_clients
    - not_converted_clients
)


result_total = pd.DataFrame({
    'Клиенты с просмотром оффера': [total_view_clients],
    'Не оформили подписку после просмотра': [not_converted_clients],
    'Оформили подписку после просмотра': [converted_clients],
    'Доля без подписки после просмотра, %': [
        not_converted_clients
        / total_view_clients
        * 100
    ]
})

display(
    result_total.style.format({
        'Клиенты с просмотром оффера': '{:,.0f}',
        'Не оформили подписку после просмотра': '{:,.0f}',
        'Оформили подписку после просмотра': '{:,.0f}',
        'Доля без подписки после просмотра, %': '{:.1f}%'
    })
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

mobapp_act['calc_date'] = pd.to_datetime(mobapp_act['calc_date'])
subscr_status['subscr_date_act'] = pd.to_datetime(subscr_status['subscr_date_act'])

first_view = (
    mobapp_act[mobapp_act['metric_id'] == 1]
    .groupby('magnit_id', as_index=False)
    .agg(first_view_date=('calc_date', 'min'))
)

subscr_min = (
    subscr_status
    .groupby(['contact_id', 'magnit_id'], as_index=False)
    .agg(first_subscr_date=('subscr_date_act', 'min'))
)

base = (
    client_cohorts
    .merge(subscr_min, left_on='client_id', right_on='contact_id', how='left')
    .merge(first_view, on='magnit_id', how='left')
)

base['saw_offer'] = base['first_view_date'].notna()

base['not_converted_after_view'] = (
    base['saw_offer']
    & (
        base['first_subscr_date'].isna()
        | (base['first_subscr_date'] <= base['first_view_date'])
    )
)

cohort_result = (
    base
    .groupby('campaigns_cnt', as_index=False)
    .agg(
        total_clients=('client_id', 'nunique'),
        saw_offer_clients=('saw_offer', 'sum'),
        not_saw_offer_clients=('saw_offer', lambda x: (~x).sum()),
        not_converted_clients=('not_converted_after_view', 'sum')
    )
)

cohort_result['saw_offer_pct'] = (
    cohort_result['saw_offer_clients'] / cohort_result['total_clients'] * 100
).round(1)

cohort_result['not_saw_offer_pct'] = (
    cohort_result['not_saw_offer_clients'] / cohort_result['total_clients'] * 100
).round(1)

cohort_result['not_converted_pct_from_viewers'] = (
    cohort_result['not_converted_clients'] / cohort_result['saw_offer_clients'] * 100
).round(1)

cohort_result

In [ ]:
cohort_table = cohort_result.rename(columns={
    'campaigns_cnt': 'Количество кампаний',
    'total_clients': 'Всего клиентов',
    'saw_offer_clients': 'Видели оффер',
    'not_saw_offer_clients': 'Не видели оффер',
    'not_converted_clients': 'Видели, но не оформили подписку',
    'saw_offer_pct': 'Доля видевших, %',
    'not_saw_offer_pct': 'Доля не видевших, %',
    'not_converted_pct_from_viewers': 'Доля без подписки среди видевших, %'
})

display(
    cohort_table.style.format({
        'Всего клиентов': '{:,.0f}',
        'Видели оффер': '{:,.0f}',
        'Не видели оффер': '{:,.0f}',
        'Видели, но не оформили подписку': '{:,.0f}',
        'Доля видевших, %': '{:.1f}%',
        'Доля не видевших, %': '{:.1f}%',
        'Доля без подписки среди видевших, %': '{:.1f}%'
    })
)